# an early notebook — Toy Jacobian walkthrough (runs anywhere, no CHPC)

A **3-unit LSTM with 4 inputs over 8 days**, small enough that every matrix in the paper's
Section 2.3 can be printed and read. The real model is identical with $H=256$, $D=32$, $T=365$.

Notation (PyTorch / NeuralHydrology `CudaLSTM` convention):

| symbol | shape (toy) | shape (real) | meaning |
|---|---|---|---|
| $x_t$ | 4 | 32 | input on day $t$: dynamic forcings followed by static attributes (statics repeated every day) |
| $h_t,\ c_t$ | 3 | 256 | hidden and cell state |
| $s_t=[h_t;c_t]$ | 6 | 512 | joint state |
| $W_{ih}$ | 12×4 | 1024×32 | input weights, rows stacked as $[i;f;g;o]$ |
| $W_{hh}$ | 12×3 | 1024×256 | recurrent weights |
| $b$ | 12 | 1024 | bias ($b_{ih}+b_{hh}$) |
| $w_{out},\ b_{out}$ | 3, 1 | 256, 1 | linear head, reads $h_T$ on the last day only |

**LSTM equations** (one day):
$$a_t = W_{ih}x_t + W_{hh}h_{t-1} + b,\qquad [a^i_t;a^f_t;a^g_t;a^o_t]=a_t$$
$$i_t=\sigma(a^i_t),\quad f_t=\sigma(a^f_t),\quad g_t=\tanh(a^g_t),\quad o_t=\sigma(a^o_t)$$
$$c_t = f_t\odot c_{t-1} + i_t\odot g_t,\qquad h_t = o_t\odot\tanh(c_t),\qquad \hat y_T = w_{out}^\top h_T + b_{out}$$


In [ ]:
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True, linewidth=140)
pd.set_option("display.precision", 4); pd.set_option("display.width", 140)

sig  = lambda a: 1/(1+np.exp(-a))
dsig = lambda a: sig(a)*(1-sig(a))          # sigma'(a) = sigma(1-sigma)
dtanh= lambda a: 1-np.tanh(a)**2            # tanh'(a) = 1 - tanh^2

H, D, T = 3, 4, 8
DYN  = ["prcp", "tmax"]                      # dynamic forcings (real model: 5)
STAT = ["aridity", "frac_snow"]              # static attributes (real model: 27)
INPUTS = DYN + STAT
UNITS  = [f"u{i}" for i in range(H)]
GATES  = ["i", "f", "g", "o"]

## 1. The input matrix $X\in\mathbb{R}^{T\times D}$

One row per day. The static columns are **the same number every day** — that is why the
sensitivity to a static attribute is a *sum over all days* (Section 7 below).

In [ ]:
rng = np.random.default_rng(3)
days = pd.date_range("2000-04-01", periods=T, freq="D")
X = np.zeros((T, D))
X[:, 0] = [0.0, 0.0, 2.1, 0.4, 0.0, 0.0, 1.3, 0.0]      # prcp (standardized units): two storms
X[:, 1] = [-0.8, -0.5, -0.2, 0.1, 0.4, 0.7, 0.9, 1.1]   # tmax: warming through the week
X[:, 2] = 0.9                                            # aridity (static, repeated)
X[:, 3] = -0.6                                           # frac_snow (static, repeated)
Xdf = pd.DataFrame(X, index=days.strftime("%m-%d"), columns=INPUTS)
Xdf

## 2. The weights

Rows of $W_{ih}$, $W_{hh}$ and $b$ are stacked gate-by-gate: rows $0..H{-}1$ are the **input gate** $i$,
then **forget** $f$, **cell candidate** $g$, **output** $o$. `np.split(a, 4)` recovers them.

The forget-gate biases are set so the three units have deliberately different memory:
u0 fast, u1 slow, u2 in between. (In the real model, training sets these.)

In [ ]:
W_ih = rng.normal(0, 0.6, (4*H, D))
W_hh = rng.normal(0, 0.4, (4*H, H))
b    = np.zeros(4*H)
b[H:2*H] = [-1.0, 3.0, 1.0]        # forget-gate bias -> f ~ 0.27, 0.95, 0.73 at rest
w_out = np.array([0.8, 0.5, -0.2]) # head weights: u0 read most, u2 least
b_out = 0.1

rows = [f"{g}_{u}" for g in GATES for u in UNITS]
display(pd.DataFrame(W_ih, index=rows, columns=INPUTS).style.set_caption("W_ih  [4H x D]"))
display(pd.DataFrame(W_hh, index=rows, columns=UNITS).style.set_caption("W_hh  [4H x H]"))
print("b    =", b, "\nw_out=", w_out, " b_out=", b_out)

## 3. Forward pass (the "unroll")

We store every gate value because **all Jacobians below are closed-form functions of these
stored activations** — nothing has to be re-run. This is exactly what `unroll_np` does in `jlens_utils.py`.

In [ ]:
def unroll(X, W_ih, W_hh, b, w_out, b_out, h0=None, c0=None):
    T = X.shape[0]; H = W_hh.shape[1]
    h = np.zeros(H) if h0 is None else h0.copy()
    c = np.zeros(H) if c0 is None else c0.copy()
    S = {k: np.zeros((T, H)) for k in ["i", "f", "g", "o", "c", "h", "c_prev", "h_prev"]}
    for t in range(T):
        S["c_prev"][t], S["h_prev"][t] = c, h
        a = W_ih @ X[t] + W_hh @ h + b
        ai, af, ag, ao = np.split(a, 4)
        i, f, g, o = sig(ai), sig(af), np.tanh(ag), sig(ao)
        c = f*c + i*g
        h = o*np.tanh(c)
        for k, v in zip(["i","f","g","o","c","h"], [i,f,g,o,c,h]): S[k][t] = v
    y = w_out @ h + b_out
    return y, S

y, S = unroll(X, W_ih, W_hh, b, w_out, b_out)
print(f"prediction on last day  y_T = {y:.4f}\n")
tbl = pd.concat({k: pd.DataFrame(S[k], index=Xdf.index, columns=UNITS) for k in ["f","i","c","h"]}, axis=1)
tbl

## 4. Per-unit memory timescale from the forget gate

Ignore the cross-unit terms for a moment. Each unit's cell state obeys
$c_{t,j} = f_{t,j}\,c_{t-1,j} + (\text{new input})$. A perturbation injected into unit $j$ on day
$t{-}k$ survives to day $t$ multiplied by $\prod f \approx f_j^{\,k} = e^{-k/\tau_j}$, so

$$\boxed{\ \tau_j = -\dfrac{\Delta t}{\ln f_j}\ }\qquad(\Delta t = 1\text{ day; Tallec \& Ollivier 2018})$$

This is a linear reservoir with residence time $\tau_j$. The **readout-weighted** summary in the paper is
$$\overline{\log_{10}\tau} = \frac{\sum_j |w_{out,j}|\,\log_{10}\tau_j}{\sum_j |w_{out,j}|},$$
and the "quick / event / seasonal / integrator" fractions are the share of $\sum|w_{out,j}|$ in each $\tau$ band.

In [ ]:
tau_t = -1/np.log(np.clip(S["f"], 1e-12, 1-1e-12))              # [T, H] a timescale per unit per day
tau_u = np.median(tau_t, axis=0)                                   # per-unit, median over days (paper)
w_abs = np.abs(w_out)
tau_readout_log10 = (w_abs * np.log10(tau_u)).sum() / w_abs.sum()
BANDS = {"quick": (0, 3), "event": (3, 30), "seasonal": (30, 365), "integrator": (365, np.inf)}
frac = {k: w_abs[(tau_u >= lo) & (tau_u < hi)].sum()/w_abs.sum() for k, (lo, hi) in BANDS.items()}

display(pd.DataFrame(tau_t, index=Xdf.index, columns=UNITS).style.set_caption("tau_t = -1/ln f_t  [days]"))
print("per-unit tau (median over days):", dict(zip(UNITS, tau_u.round(2))))
print(f"readout-weighted mean log10 tau = {tau_readout_log10:.3f}  ->  {10**tau_readout_log10:.2f} days")
print("readout fraction by band:", {k: round(v, 2) for k, v in frac.items()})

## 5. The state-transition Jacobian $J_t=\dfrac{\partial s_t}{\partial s_{t-1}}\in\mathbb{R}^{2H\times 2H}$ (retention)

With $s=[h;c]$ this is a 2×2 block matrix. Write $\phi_t=\tanh(c_t)$ and let $W^{\,i}_{hh},W^{\,f}_{hh},\dots$ be the gate blocks of $W_{hh}$.

$$
J_t=\begin{bmatrix}
\dfrac{\partial h_t}{\partial h_{t-1}} & \dfrac{\partial h_t}{\partial c_{t-1}}\\[2ex]
\dfrac{\partial c_t}{\partial h_{t-1}} & \dfrac{\partial c_t}{\partial c_{t-1}}
\end{bmatrix}
$$

Bottom-right (**exact diagonal** — the gates see $h_{t-1}$, not $c_{t-1}$):
$$\frac{\partial c_t}{\partial c_{t-1}} = \mathrm{diag}(f_t)$$

Bottom-left (how yesterday's hidden state re-steers today's gates):
$$\frac{\partial c_t}{\partial h_{t-1}} = \mathrm{diag}\big(c_{t-1}\odot f_t(1{-}f_t)\big)W^{f}_{hh} + \mathrm{diag}\big(g_t\odot i_t(1{-}i_t)\big)W^{i}_{hh} + \mathrm{diag}\big(i_t\odot(1{-}g_t^2)\big)W^{g}_{hh}$$

Top row, via $h_t=o_t\odot\phi_t$ and the chain rule through $c_t$:
$$\frac{\partial h_t}{\partial c_t}=\mathrm{diag}\big(o_t\odot(1{-}\phi_t^2)\big),\qquad
\frac{\partial h_t}{\partial c_{t-1}} = \frac{\partial h_t}{\partial c_t}\,\mathrm{diag}(f_t)$$
$$\frac{\partial h_t}{\partial h_{t-1}} = \mathrm{diag}\big(\phi_t\odot o_t(1{-}o_t)\big)W^{o}_{hh} + \frac{\partial h_t}{\partial c_t}\,\frac{\partial c_t}{\partial h_{t-1}}$$

The paper's "per-unit linear reservoir" reading keeps **only** $\mathrm{diag}(f_t)$; the three other blocks are the
cross-unit coupling that the full-matrix eigenvalue analysis retains.

In [ ]:
def gate_blocks(W):
    return dict(zip(GATES, np.split(W, 4, axis=0)))

def state_jacobian(S, t, W_hh):
    Wb = gate_blocks(W_hh)
    i, f, g, o, c, cp = (S[k][t] for k in ["i","f","g","o","c","c_prev"])
    phi = np.tanh(c)
    dc_dc   = np.diag(f)
    dc_dh   = (np.diag(cp*f*(1-f)) @ Wb["f"] + np.diag(g*i*(1-i)) @ Wb["i"] + np.diag(i*(1-g**2)) @ Wb["g"])
    dh_dc_t = np.diag(o*(1-phi**2))
    dh_dc   = dh_dc_t @ dc_dc
    dh_dh   = np.diag(phi*o*(1-o)) @ Wb["o"] + dh_dc_t @ dc_dh
    return np.block([[dh_dh, dh_dc], [dc_dh, dc_dc]])          # rows = [h_t; c_t], cols = [h_{t-1}; c_{t-1}]

SLAB = [f"h_{u}" for u in UNITS] + [f"c_{u}" for u in UNITS]
t = 2   # the big storm day
J = state_jacobian(S, t, W_hh)
display(pd.DataFrame(J, index=[s+"(t)" for s in SLAB], columns=[s+"(t-1)" for s in SLAB])
          .style.set_caption(f"J_t = ds_t/ds_(t-1) on day {Xdf.index[t]}  (bottom-right block = diag(f_t))"))
print("diag of bottom-right block:", np.diag(J[H:, H:]).round(4), "  f_t =", S["f"][t].round(4))

### Check it against finite differences

Perturb one entry of $s_{t-1}$, take one LSTM step, and difference. If the closed form is right the two agree to ~1e-6.

In [ ]:
def one_step(x, h, c, W_ih, W_hh, b):
    ai, af, ag, ao = np.split(W_ih @ x + W_hh @ h + b, 4)
    c_new = sig(af)*c + sig(ai)*np.tanh(ag)
    return np.concatenate([sig(ao)*np.tanh(c_new), c_new])

def fd_state_jacobian(S, t, X, eps=1e-6):
    s0 = np.concatenate([S["h_prev"][t], S["c_prev"][t]])
    Jfd = np.zeros((2*H, 2*H))
    for j in range(2*H):
        sp, sm = s0.copy(), s0.copy(); sp[j] += eps; sm[j] -= eps
        Jfd[:, j] = (one_step(X[t], sp[:H], sp[H:], W_ih, W_hh, b) - one_step(X[t], sm[:H], sm[H:], W_ih, W_hh, b)) / (2*eps)
    return Jfd

Jfd = fd_state_jacobian(S, t, X)
print("max |closed-form - finite-diff| =", np.abs(J - Jfd).max())
assert np.abs(J - Jfd).max() < 1e-6

### Eigenvalues: the full matrix vs the diagonal reading

The eigenvalues $\lambda$ of $J_t$ are the true per-step retention factors of the coupled system; $|\lambda|\to1$ is a
non-decaying mode. The paper counts slow ($0.99\le|\lambda|<1$) and non-decaying ($|\lambda|\ge1$) modes. Compare with the
forget gates, which are the eigenvalues you would get if the off-diagonal blocks were zero.

In [ ]:
lam = np.linalg.eigvals(J)
print("eigenvalues |lambda| of full J_t :", np.sort(np.abs(lam))[::-1].round(4))
print("forget gates f_t (diagonal view) :", np.sort(S['f'][t])[::-1].round(4))
print("implied tau from |lambda|        :", (-1/np.log(np.clip(np.sort(np.abs(lam))[::-1], 1e-12, 1-1e-12))).round(2))

## 6. The input Jacobian $E_t=\dfrac{\partial s_t}{\partial x_t}\in\mathbb{R}^{2H\times D}$ (encoding)

Same derivation with $W_{ih}$ in place of $W_{hh}$ (the input enters the gates the same way $h_{t-1}$ does):
$$\frac{\partial c_t}{\partial x_t} = \mathrm{diag}\big(c_{t-1}\odot f_t(1{-}f_t)\big)W^{f}_{ih} + \mathrm{diag}\big(g_t\odot i_t(1{-}i_t)\big)W^{i}_{ih} + \mathrm{diag}\big(i_t\odot(1{-}g_t^2)\big)W^{g}_{ih}$$
$$\frac{\partial h_t}{\partial x_t} = \mathrm{diag}\big(\phi_t\odot o_t(1{-}o_t)\big)W^{o}_{ih} + \frac{\partial h_t}{\partial c_t}\frac{\partial c_t}{\partial x_t}$$

It is **state-dependent** (through $c_{t-1}$ and the gate slopes), which is why the paper says the encoding is
"not a fixed weight matrix": the same millimetre of rain is written into memory differently on a wet day and a dry day.

In [ ]:
def input_jacobian(S, t, W_ih):
    Wb = gate_blocks(W_ih)
    i, f, g, o, c, cp = (S[k][t] for k in ["i","f","g","o","c","c_prev"])
    phi = np.tanh(c)
    dc_dx = (np.diag(cp*f*(1-f)) @ Wb["f"] + np.diag(g*i*(1-i)) @ Wb["i"] + np.diag(i*(1-g**2)) @ Wb["g"])
    dh_dx = np.diag(phi*o*(1-o)) @ Wb["o"] + np.diag(o*(1-phi**2)) @ dc_dx
    return np.vstack([dh_dx, dc_dx])

E = input_jacobian(S, t, W_ih)
display(pd.DataFrame(E, index=SLAB, columns=INPUTS).style.set_caption(f"E_t = ds_t/dx_t on day {Xdf.index[t]}"))

# finite-difference check
Efd = np.zeros_like(E); eps = 1e-6
for j in range(D):
    xp, xm = X[t].copy(), X[t].copy(); xp[j] += eps; xm[j] -= eps
    Efd[:, j] = (one_step(xp, S["h_prev"][t], S["c_prev"][t], W_ih, W_hh, b) - one_step(xm, S["h_prev"][t], S["c_prev"][t], W_ih, W_hh, b)) / (2*eps)
print("max |closed-form - finite-diff| =", np.abs(E - Efd).max()); assert np.abs(E - Efd).max() < 1e-6

## 7. The readout and the full chain: $\dfrac{\partial \hat y_T}{\partial x_{T-k}}$

The head reads only $h_T$, so as a row vector over $s=[h;c]$:
$$\frac{\partial \hat y_T}{\partial s_T} = [\,w_{out}^\top,\ \mathbf{0}\,]\in\mathbb{R}^{1\times 2H}$$

Chain rule along the sequence (the paper's Eq. 1):
$$\boxed{\ \frac{\partial \hat y_T}{\partial x_{T-k}} = \underbrace{\frac{\partial \hat y_T}{\partial s_T}}_{1\times 2H}\ \underbrace{J_T\,J_{T-1}\cdots J_{T-k+1}}_{2H\times 2H\ \text{(k factors)}}\ \underbrace{E_{T-k}}_{2H\times D}\ \in\mathbb{R}^{1\times D}\ }$$

Read right-to-left: the input on day $T{-}k$ is **encoded** into the state, **retained** (and mixed) through $k$ daily
transitions, then **decoded** by the head. For $k=0$ there is no transition product: $\partial\hat y_T/\partial x_T = [w_{out}^\top,0]\,E_T$.

The *lag profile* for precipitation is $|\partial\hat y_T/\partial P_{T-k}|$ versus $k$; its median lag is the paper's $M_{50}$.

In [ ]:
def lag_sensitivity(S, X, W_ih, W_hh, w_out):
    T = X.shape[0]
    r = np.concatenate([w_out, np.zeros(H)])           # dy/ds_T  (row vector)
    G = np.zeros((T, D))                                # G[T-k] = dy_T / dx_{T-k}
    row = r.copy()
    for t in range(T-1, -1, -1):                        # walk backwards, accumulating the product
        G[t] = row @ input_jacobian(S, t, W_ih)         # ... J_{t+1} E_t
        row = row @ state_jacobian(S, t, W_hh)          # now row = dy/ds_{t-1}
    return G

G = lag_sensitivity(S, X, W_ih, W_hh, w_out)
Gdf = pd.DataFrame(G, index=Xdf.index, columns=INPUTS)
Gdf["lag k"] = np.arange(T-1, -1, -1)
display(Gdf.style.set_caption("dy_T / dx_{T-k}   (one row per input day; last row is lag 0)"))

# finite-difference check of the WHOLE chain: perturb X[t, j], re-run the full unroll
Gfd = np.zeros_like(G); eps = 1e-6
for t in range(T):
    for j in range(D):
        Xp, Xm = X.copy(), X.copy(); Xp[t, j] += eps; Xm[t, j] -= eps
        Gfd[t, j] = (unroll(Xp, W_ih, W_hh, b, w_out, b_out)[0] - unroll(Xm, W_ih, W_hh, b, w_out, b_out)[0]) / (2*eps)
print("max |chain-rule - finite-diff| =", np.abs(G - Gfd).max()); assert np.abs(G - Gfd).max() < 1e-6

prof = np.abs(G[::-1, 0])                               # |dy_T/dP_{T-k}|, index = lag k
cdf = np.cumsum(prof)/prof.sum()
print("\nprecip lag profile |dy_T/dP_{T-k}|:", prof.round(4))
print("M50 (lag at which 50% of cumulative sensitivity is reached):", int(np.searchsorted(cdf, 0.5)), "days")

## 8. Does the diagonal (per-unit $\tau$) reading reproduce the exact chain?

Replace every $J_t$ by its forget-gate diagonal only, $\tilde J_t = \mathrm{diag}(0,\,f_t)$ padded so that the $h$ block
is passed through $\partial h_t/\partial c_t$ at the end. Compare the resulting precipitation lag profile with the exact one.
Where they diverge, cross-unit coupling matters and the per-unit reservoir picture is only a first-order story —
this is the comparison the paper's eigenvalue counts and the propagator profile (nb08) are about.

In [ ]:
def lag_sensitivity_diag(S, X, W_ih, W_hh, w_out):
    T = X.shape[0]
    G = np.zeros((T, D))
    for k in range(T):
        t0 = T-1-k
        E = input_jacobian(S, t0, W_ih)
        if k == 0:
            G[t0] = np.concatenate([w_out, np.zeros(H)]) @ E
        else:
            # keep only the cell-state write from E, retain through diag(f) products, decode via dh_T/dc_T
            retain = np.prod(S["f"][t0+1:T], axis=0)                # prod_{j=t0+1}^{T} f_j   per unit
            phi_T = np.tanh(S["c"][T-1]); o_T = S["o"][T-1]
            G[t0] = (w_out * o_T*(1-phi_T**2) * retain) @ E[H:]      # [1xH] @ dc_{t0}/dx
    return G

Gd = lag_sensitivity_diag(S, X, W_ih, W_hh, w_out)
cmp = pd.DataFrame({"lag k": np.arange(T)[::-1], "exact |dy/dP|": np.abs(G[:, 0]), "diag-only |dy/dP|": np.abs(Gd[:, 0])}, index=Xdf.index)
cmp

## 9. Sensitivity to a static attribute

A static attribute $a$ is the same number in every $x_t$, so by the chain rule its total derivative is the **sum of the
per-day sensitivities** over the whole window:
$$\frac{d\hat y_T}{da} = \sum_{t=1}^{T}\frac{\partial \hat y_T}{\partial x_{t,a}}$$
The paper's *dynamics-sensitivity gradient* applies the same idea to $\overline{\log_{10}\tau}$ instead of $\hat y_T$
(how much does the memory portfolio move if aridity moves?). Finite-difference check: shift the whole static column.

In [ ]:
for j, name in enumerate(STAT, start=len(DYN)):
    chain = G[:, j].sum()
    Xp, Xm = X.copy(), X.copy(); Xp[:, j] += eps; Xm[:, j] -= eps
    fd = (unroll(Xp, W_ih, W_hh, b, w_out, b_out)[0] - unroll(Xm, W_ih, W_hh, b, w_out, b_out)[0]) / (2*eps)
    print(f"d y_T / d {name:10s}: chain-rule sum = {chain:+.5f}   finite-diff = {fd:+.5f}")

## 10. Optional: the same thing with PyTorch autograd (needs `torch`)

This is how `lag_profile_input` in `jlens_utils.py` works. It should reproduce `G` above to ~1e-12 in float64.

In [ ]:
try:
    import torch
    Xt = torch.tensor(X, dtype=torch.float64, requires_grad=True)
    Wi, Wh, bt, wo = (torch.tensor(a, dtype=torch.float64) for a in (W_ih, W_hh, b, w_out))
    h = torch.zeros(H, dtype=torch.float64); c = torch.zeros(H, dtype=torch.float64)
    for t in range(T):
        ai, af, ag, ao = (Wi @ Xt[t] + Wh @ h + bt).chunk(4)
        c = torch.sigmoid(af)*c + torch.sigmoid(ai)*torch.tanh(ag)
        h = torch.sigmoid(ao)*torch.tanh(c)
    yt = wo @ h + b_out
    yt.backward()
    print("max |autograd - chain rule| =", np.abs(Xt.grad.numpy() - G).max())
except ImportError:
    print("torch not installed - skip (the finite-difference checks above already validate G)")

## What to carry to the real model

| toy | real (`jlens_utils.Model`) |
|---|---|
| `X` [8×4] | `build_X` → [B, 365, 32] windows, 45-day stride |
| `unroll` | `unroll_np` (gates) / `unroll_torch` (autograd) |
| `tau_u`, `tau_readout_log10`, `frac` | `portfolio()` |
| `G[:,0]` → `prof`, `M50` | `lag_profile_input` + `profile_stats` |
| `state_jacobian` eigenvalues | the slow/non-decaying mode counts (8 windows × 4 days) |
| Section 8 diag-vs-exact | `state_propagator_profile` (nb08) |
| Section 9 | `dynamics_grad` sweep (an early notebook) |
